# Bloomberg data upload

Run this on the Bloomberg computer with Bloomberg Terminal open and logged in.

Order: packages, app setup, connection test, upload test, Bloomberg test, then real data.

## 1. Packages

Run this first. If install is blocked, ask IT for these packages in Jupyter: `requests`, `pandas`, `pyarrow`, `xbbg`.

In [ ]:
import importlib.util
import subprocess
import sys

def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is not None:
        print(f"OK: {import_name}")
        return True
    print(f"Installing {package_name}...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        print(f"OK: installed {package_name}")
        return True
    except Exception as exc:
        print(f"FAILED: could not install {package_name}: {exc}")
        return False

PACKAGE_STATUS = {
    "requests": ensure_package("requests"),
    "pandas": ensure_package("pandas"),
    "pyarrow": ensure_package("pyarrow"),
    "xbbg": ensure_package("xbbg"),
}
PACKAGE_STATUS

## 2. App setup

Set the app URL. Enter the bridge key in the prompt.

In [ ]:
import getpass
import os

APP_DOMAIN = "https://YOUR_PUBLIC_APP_DOMAIN"
BRIDGE_ID = "supervisor-bloomberg-terminal-01"

os.environ["BT_BLOOMBERG_ENDPOINT"] = APP_DOMAIN.rstrip("/")
os.environ["BT_BLOOMBERG_BRIDGE_ID"] = BRIDGE_ID
os.environ["BT_BLOOMBERG_BRIDGE_KEY"] = getpass.getpass("Bridge key: ")

ENDPOINT = os.environ["BT_BLOOMBERG_ENDPOINT"].rstrip("/")
BRIDGE_ID = os.environ["BT_BLOOMBERG_BRIDGE_ID"]
BRIDGE_KEY = os.environ["BT_BLOOMBERG_BRIDGE_KEY"]

assert ENDPOINT.startswith("https://"), "Use the public HTTPS app URL"
assert BRIDGE_KEY, "Bridge key is required"
print("Configured endpoint:", ENDPOINT)
print("Bridge id:", BRIDGE_ID)

## 3. Connection test

Expected result: HTTP 200.

In [ ]:
import json
import requests

def bridge_headers():
    return {
        "X-Bloomberg-Bridge-Key": BRIDGE_KEY,
        "X-Bloomberg-Bridge-Id": BRIDGE_ID,
    }

response = requests.get(f"{ENDPOINT}/bridge/bloomberg/health", headers=bridge_headers(), timeout=30)
print("HTTP", response.status_code)
print(response.text[:2000])
response.raise_for_status()

## 4. Upload functions

Run once before any upload cells.

In [ ]:
from datetime import datetime, timezone
from io import BytesIO
import hashlib
import uuid

import pandas as pd

def to_parquet_bytes(frame):
    buffer = BytesIO()
    frame.to_parquet(buffer, index=False)
    return buffer.getvalue()

def sha256_hex(payload):
    return hashlib.sha256(payload).hexdigest()

def make_request_id(source):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    return f"{source}-{stamp}-{uuid.uuid4().hex[:8]}"

def upload_frame(
    frame,
    *,
    source,
    kind,
    securities,
    fields,
    start_date=None,
    end_date=None,
    periodicity=None,
    overrides=None,
    timeout=120,
):
    if frame is None or frame.empty:
        raise ValueError("Refusing to upload an empty dataframe")
    clean = frame.copy()
    clean.columns = [str(column) for column in clean.columns]
    payload = to_parquet_bytes(clean)
    request_id = make_request_id(source)
    digest = sha256_hex(payload)
    manifest = {
        "schema_version": 1,
        "bridge_id": BRIDGE_ID,
        "request_id": request_id,
        "bloomberg_source": source,
        "kind": kind,
        "securities": list(securities),
        "fields": list(fields),
        "start_date": start_date,
        "end_date": end_date,
        "periodicity": periodicity,
        "overrides": overrides or {},
        "row_count": int(len(clean)),
        "columns": [str(column) for column in clean.columns],
        "data_sha256": digest,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    files = {"file": (f"{request_id}.parquet", payload, "application/octet-stream")}
    data = {"manifest_json": json.dumps(manifest, separators=(",", ":"))}
    response = requests.post(
        f"{ENDPOINT}/bridge/bloomberg/batches",
        headers=bridge_headers(),
        data=data,
        files=files,
        timeout=timeout,
    )
    print("HTTP", response.status_code)
    print(response.text[:3000])
    response.raise_for_status()
    return response.json()

print("Upload helpers are ready")

## 5. Upload test

Run this before using Bloomberg data.

In [ ]:
mock_frame = pd.DataFrame(
    [
        {"date": "2026-05-01", "security": "ATW MA Equity", "field": "PX_LAST", "value": 500.0},
        {"date": "2026-05-04", "security": "ATW MA Equity", "field": "PX_LAST", "value": 502.5},
        {"date": "2026-05-05", "security": "ATW MA Equity", "field": "PX_LAST", "value": 501.0},
    ]
)
LAST_UPLOAD = upload_frame(
    mock_frame,
    source="bdh",
    kind="time_series",
    securities=["ATW MA Equity"],
    fields=["PX_LAST"],
    start_date="2026-05-01",
    end_date="2026-05-05",
    periodicity="DAILY",
)
LAST_UPLOAD

## 6. Bloomberg test

Bloomberg Terminal must be open and logged in.

In [ ]:
try:
    from xbbg import blp
    XBBG_AVAILABLE = True
    print("OK: xbbg imported")
except Exception as exc:
    XBBG_AVAILABLE = False
    print("FAILED: xbbg import/access failed")
    print(repr(exc))

if XBBG_AVAILABLE:
    sample = blp.bdh(
        tickers=["ATW MA Equity"],
        flds=["PX_LAST"],
        start_date="2026-05-01",
        end_date="2026-05-12",
    )
    display(sample.tail())

## 7. Table cleanup

Run once before real data uploads.

In [ ]:
OHLCV_FIELDS = ["PX_OPEN", "PX_HIGH", "PX_LOW", "PX_LAST", "VOLUME"]
print("Default OHLCV fields:", OHLCV_FIELDS)

def normalize_bdh(raw):
    if raw is None or raw.empty:
        return pd.DataFrame(columns=["date", "security", "field", "value"])
    frame = raw.copy()
    frame.index = pd.to_datetime(frame.index).date.astype(str)
    if isinstance(frame.columns, pd.MultiIndex):
        long = frame.stack(list(range(frame.columns.nlevels))).reset_index()
        long.columns = ["date", "security", "field", "value"][: len(long.columns)]
        if "field" not in long.columns and len(long.columns) == 3:
            long.columns = ["date", "field", "value"]
            long["security"] = "UNKNOWN"
    else:
        long = frame.reset_index().melt(id_vars=frame.index.name or "index", var_name="field", value_name="value")
        long = long.rename(columns={frame.index.name or "index": "date"})
        long["security"] = "UNKNOWN"
    long = long[["date", "security", "field", "value"]]
    long = long.dropna(subset=["value"])
    long["value"] = pd.to_numeric(long["value"], errors="ignore")
    return long

def normalize_bdib(raw, security, field="PX_LAST"):
    if raw is None or raw.empty:
        return pd.DataFrame(columns=["datetime", "security", "field", "value"])
    frame = raw.copy().reset_index()
    datetime_column = "time" if "time" in frame.columns else frame.columns[0]
    value_column = "close" if "close" in frame.columns else frame.columns[-1]
    long = pd.DataFrame(
        {
            "datetime": pd.to_datetime(frame[datetime_column]).astype(str),
            "security": security,
            "field": field,
            "value": pd.to_numeric(frame[value_column], errors="coerce"),
        }
    ).dropna(subset=["value"])
    return long

print("Normalization helpers are ready")

## 8. Upload one daily series

Start with one security.

In [ ]:
if not XBBG_AVAILABLE:
    raise RuntimeError("xbbg is not available in this Jupyter environment")

SECURITIES = ["ATW MA Equity"]
FIELDS = OHLCV_FIELDS.copy()
START_DATE = "2024-01-01"
END_DATE = "2026-05-12"

raw_daily = blp.bdh(tickers=SECURITIES, flds=FIELDS, start_date=START_DATE, end_date=END_DATE)
daily_frame = normalize_bdh(raw_daily)
display(daily_frame.tail())
print("Rows:", len(daily_frame))

LAST_UPLOAD = upload_frame(
    daily_frame,
    source="bdh",
    kind="time_series",
    securities=SECURITIES,
    fields=FIELDS,
    start_date=START_DATE,
    end_date=END_DATE,
    periodicity="DAILY",
)
LAST_UPLOAD

## 9. Check MASI tickers

Edit `MASI_SYMBOLS` if needed. This checks `MA Equity` and `MC Equity`.

In [ ]:
MASI_SYMBOLS = [
    "ATW",
    "BCP",
    "IAM",
    "BOA",
    "LHM",
]

def probe_equity_candidates(symbols, suffixes=("MA Equity", "MC Equity"), field="PX_LAST"):
    rows = []
    for symbol in symbols:
        for suffix in suffixes:
            ticker = f"{symbol} {suffix}"
            try:
                probe = blp.bdh(tickers=[ticker], flds=[field], start_date="2026-05-01", end_date="2026-05-12")
                available = probe is not None and not probe.dropna(how="all").empty
                rows.append({"symbol": symbol, "ticker": ticker, "available": bool(available), "error": None})
            except Exception as exc:
                rows.append({"symbol": symbol, "ticker": ticker, "available": False, "error": str(exc)[:300]})
    return pd.DataFrame(rows)

availability = probe_equity_candidates(MASI_SYMBOLS)
display(availability)
AVAILABLE_MASI_TICKERS = availability.loc[availability["available"], "ticker"].tolist()
AVAILABLE_MASI_TICKERS

## 10. Upload MASI daily data

Only tickers that passed the check are uploaded.

In [ ]:
if not AVAILABLE_MASI_TICKERS:
    raise RuntimeError("No MASI tickers passed the availability probe")

MASI_FIELDS = OHLCV_FIELDS.copy()
MASI_START_DATE = "2020-01-01"
MASI_END_DATE = "2026-05-12"

raw_masi = blp.bdh(
    tickers=AVAILABLE_MASI_TICKERS,
    flds=MASI_FIELDS,
    start_date=MASI_START_DATE,
    end_date=MASI_END_DATE,
)
masi_daily_frame = normalize_bdh(raw_masi)
display(masi_daily_frame.tail())
print("Rows:", len(masi_daily_frame))

LAST_UPLOAD = upload_frame(
    masi_daily_frame,
    source="bdh",
    kind="time_series",
    securities=AVAILABLE_MASI_TICKERS,
    fields=MASI_FIELDS,
    start_date=MASI_START_DATE,
    end_date=MASI_END_DATE,
    periodicity="DAILY",
    timeout=300,
)
LAST_UPLOAD

## 11. Optional intraday data

Test one ticker and one date first.

In [ ]:
INTRADAY_TICKER = "ATW MA Equity"
INTRADAY_DATE = "2026-05-12"
INTRADAY_INTERVAL_MINUTES = 60

raw_intraday = blp.bdib(ticker=INTRADAY_TICKER, dt=INTRADAY_DATE, interval=INTRADAY_INTERVAL_MINUTES)
intraday_frame = normalize_bdib(raw_intraday, INTRADAY_TICKER, field="PX_LAST")
display(intraday_frame.tail())
print("Rows:", len(intraday_frame))

if intraday_frame.empty:
    print("No intraday data returned for this ticker/date/interval. Try another date or interval.")
else:
    LAST_UPLOAD = upload_frame(
        intraday_frame,
        source="bdib",
        kind="time_series",
        securities=[INTRADAY_TICKER],
        fields=["PX_LAST"],
        start_date=INTRADAY_DATE,
        end_date=INTRADAY_DATE,
        periodicity=f"INTRADAY_{INTRADAY_INTERVAL_MINUTES}",
        timeout=300,
    )
    display(LAST_UPLOAD)

## 12. Optional BQL

Run this only if BQL works in this Jupyter setup.

In [ ]:
try:
    import bql
    service = bql.Service()
    bql_response = service.execute("get(px_last) for(['ATW MA Equity'])")
    bql_tables = [item.df().reset_index() for item in bql_response]
    bql_frame = pd.concat(bql_tables, ignore_index=True) if bql_tables else pd.DataFrame()
    display(bql_frame.head())
    if not bql_frame.empty:
        LAST_UPLOAD = upload_frame(
            bql_frame,
            source="bql",
            kind="bql_table",
            securities=["ATW MA Equity"],
            fields=["px_last"],
            timeout=300,
        )
        display(LAST_UPLOAD)
except Exception as exc:
    print("BQL probe failed or BQL is unavailable:")
    print(repr(exc))

## Troubleshooting

- HTTP 401 or 403: wrong bridge key.
- Timeout: network or domain block.
- `xbbg` fails: Bloomberg Python access is missing.
- Empty data: ticker, field, date range, or entitlement issue.
- Large upload fails: use fewer tickers or a shorter date range.